# Logit lens on GPT-2

Project the residual stream after every layer through the final LayerNorm and unembedding,
and look at the top-1 token at each position — to see at what depth the model settles on its prediction.

## Setup

In [1]:
!git clone https://github.com/ai360-project-school-j-lens/jacobian-lens-gpt2.git

Cloning into 'jacobian-lens-gpt2'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 74 (delta 9), reused 18 (delta 5), pack-reused 38 (from 1)
Receiving objects: 100% (74/74), 3.06 MiB | 22.41 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [2]:
import json
import sys

import torch
from IPython.display import display

REPO_DIR = '/kaggle/working/jacobian-lens-gpt2'
sys.path.append(REPO_DIR)

from gpt2 import GPT2LensModel
from jlens import JacobianLens
from jlens.vis import build_page, compute_slice, notebook_iframe

In [3]:
gpt = GPT2LensModel.from_pretrained()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## Dataset

In [4]:
dataset_name = REPO_DIR + "/data/evaluations/lens-eval-order-ops.json"

dataset = json.load(open(dataset_name))

In [5]:
dataset['items'][0]

{'name': 'parens-add-mult',
 'prompt': '(2 + 3) * 4 = ',
 'target': '20',
 'intermediates': ['5', 'multiplication']}

## Sanity checks

Shapes of the encoded prompt and the hidden states, and the lens prediction at a single position.

In [6]:
ids = gpt.encode(dataset['items'][0]['prompt'])
with torch.no_grad():
    hidden_states = gpt.forward(ids, output_hidden_states=True).hidden_states

In [7]:
ids.shape

torch.Size([1, 10])

In [8]:
hidden_states[0].shape

torch.Size([1, 10, 768])

In [9]:
gpt.unembed(hidden_states[0][0, 1, :]).argmax()

tensor(7)

In [10]:
vocab = gpt.tokenizer.get_vocab()
token_to_item = {
    vocab[i] : i
    for i in vocab
}

## Logit lens visualisation

Reuses the slice page from `walkthrough.ipynb` (`compute_slice` + `build_page`): an interactive
position × layer view of token ranks. The logit lens is a Jacobian lens with `J_l = I` on every layer,
so each block output goes straight through `ln_f` + unembedding.

In [11]:
# логит-линза = J-линза с единичной матрицей на каждом слое
logit_lens = JacobianLens(
    {layer: torch.eye(gpt.d_model) for layer in range(gpt.n_layers - 1)},
    n_prompts=0,
    d_model=gpt.d_model,
)


def show_logit_lens(prompt, title="Logit lens", description=""):
    slice_data = compute_slice(gpt, logit_lens, prompt)
    page, _, _ = build_page(slice_data, prompt, title=title, description=description)
    display(notebook_iframe(page))

### Dataset prompts

In [12]:
for sample in dataset['items'][0:5]:
    show_logit_lens(
        sample['prompt'].rstrip(),
        title=sample['name'],
        description=f"target: {sample['target']!r}, intermediates: {sample['intermediates']}",
    )

### Custom prompt

In [13]:
show_logit_lens("The Eifflel Tower is located in the city of")

## Intermediate metric

For every dataset prompt, sample `k` continuations of `max_new_tokens` tokens. In each run, apply the lens
to prompt + continuation and take the top-1 token at every (layer, position). An intermediate is a *hit*
if one of its token spellings is a top-1 token somewhere in at least one of the `k` runs (pass@k).
Metric = mean over items of the fraction of intermediates that are hits.

In [14]:
from tqdm.notebook import tqdm

def intermediate_token_ids(word):
    """Single-token GPT-2 spellings of `word`: with/without leading space, as-is/lower/capitalized."""
    ids = set()
    for variant in {word, word.lower(), word.capitalize()}:
        for text in (variant, " " + variant):
            tokens = gpt.tokenizer.encode(text)
            if len(tokens) == 1:
                ids.add(tokens[0])
    if not ids:
        # слово не помещается в один токен — берём первый токен варианта с пробелом
        ids.add(gpt.tokenizer.encode(" " + word)[0])
    return ids


@torch.no_grad()
def sample_continuations(prompt, k, max_new_tokens, temperature):
    """k sampled continuations of `prompt` (text only, without the prompt)."""
    ids = gpt.encode(prompt)
    out = gpt._hf_model.generate(
        ids,
        attention_mask=torch.ones_like(ids),
        do_sample=True,
        temperature=temperature,
        top_k=0,
        max_new_tokens=max_new_tokens,
        num_return_sequences=k,
        pad_token_id=gpt.tokenizer.eos_token_id,
    )
    return gpt.tokenizer.batch_decode(out[:, ids.shape[1]:])


@torch.no_grad()
def lens_argmax(lens, text):
    """Lens top-1 token at every (layer, position): [n_layers, seq_len]; last row is the model output."""
    lens_logits, model_logits, _ = lens.apply(gpt, text)
    logits = [lens_logits[layer] for layer in sorted(lens_logits)] + [model_logits]
    # позиция 0 — добавленный <|endoftext|>, её не учитываем
    return torch.stack(logits).argmax(dim=-1)[:, 1:]


def intermediate_metric(lens, items, k, max_new_tokens=10, temperature=1.0):
    """pass@k: mean over items of the fraction of intermediates seen as lens argmax in any of k runs."""
    rows = []
    for item in tqdm(items):
        prompt = item["prompt"].rstrip()
        top1 = set()
        for continuation in sample_continuations(prompt, k, max_new_tokens, temperature):
            top1 |= set(lens_argmax(lens, prompt + continuation).flatten().tolist())
        hits = {word: bool(intermediate_token_ids(word) & top1) for word in item["intermediates"]}
        rows.append({"name": item["name"], "hits": hits, "score": sum(hits.values()) / len(hits)})
    return sum(row["score"] for row in rows) / len(rows), rows


In [16]:
k = 10

torch.manual_seed(0)
score, rows = intermediate_metric(logit_lens, dataset["items"], k, max_new_tokens=5)

print(f"\nintermediate pass@{k}: {score:.3f}")

  0%|          | 0/55 [00:00<?, ?it/s]


intermediate pass@10: 0.164


In [20]:
rows[0:5]

[{'name': 'parens-add-mult',
  'hits': {'5': True, 'multiplication': False},
  'score': 0.5},
 {'name': 'parens-sub-mult',
  'hits': {'5': True, 'multiplication': False},
  'score': 0.5},
 {'name': 'mult-parens-add',
  'hits': {'7': False, 'multiplication': False},
  'score': 0.0},
 {'name': 'mult-parens-sub',
  'hits': {'3': False, 'multiplication': False},
  'score': 0.0},
 {'name': 'add-mult-right',
  'hits': {'12': False, 'addition': False},
  'score': 0.0}]

In [21]:
sum(sum(i['hits'].values()) for i in rows) / len(rows)

0.32727272727272727